In [0]:
#imports
from pyspark.sql.window import Window
import requests
import re
import os

from bs4 import BeautifulSoup
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    IntegerType
)

# configs

BASE_URL = "https://download.bls.gov/pub/time.series/pr/"
RAW_BLS_PATH = "/Volumes/bls_dataquest/bronze/bls_dq/raw/bls/"

METADATA_TABLE = "bls_dataquest.bronze.ingestion_metadata"

HEADERS = { "User-Agent": "Databricks-Productivity-Portfolio-Project/1.0"}


import requests
import re

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.window import Window

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    IntegerType
)

# ============================================================
# 4. CREATE / VERIFY METADATA TABLE
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {METADATA_TABLE} (
    file_name STRING,
    source_url STRING,
    bls_modified_at TIMESTAMP,
    downloaded_at_utc TIMESTAMP,
    http_status INT,
    status STRING,
    error_message STRING,
    ingestion_run_id STRING
)
USING DELTA
COMMENT 'Audit metadata for BLS Productivity source file ingestion'
""")


# ============================================================
# 5. ADD change_type COLUMN IF IT DOES NOT EXIST
# ============================================================
#
# We explicitly manage the schema because Unity Catalog /
# Free Edition does not allow automatic schema migration in
# this environment.
#
# If change_type already exists, this statement will fail.
# We therefore check the existing schema first.
# ============================================================

existing_columns = {
    field.name
    for field in spark.table(
        METADATA_TABLE
    ).schema.fields
}


if "change_type" not in existing_columns:

    spark.sql(f"""
        ALTER TABLE {METADATA_TABLE}
        ADD COLUMNS (
            change_type STRING
        )
    """)

    print(
        "Added missing column: change_type"
    )

else:

    print(
        "Column already exists: change_type"
    )


print(
    f"Metadata table ready: {METADATA_TABLE}"
)


# ============================================================
# 6. GENERATE INGESTION RUN ID
# ============================================================

ingestion_run_id = datetime.now(
    timezone.utc
).strftime(
    "%Y%m%d%H%M%S%f"
)


print(
    f"Ingestion run ID: {ingestion_run_id}"
)


# ============================================================
# 7. READ EXISTING METADATA
# ============================================================

metadata = spark.table(
    METADATA_TABLE
)


# ============================================================
# 8. GET LATEST SUCCESSFUL VERSION OF EACH FILE
# ============================================================
#
# We only consider SUCCESS records.
#
# This is important because if a file failed during the
# previous run, we want the next run to retry it.
# ============================================================

window_spec = (
    Window
    .partitionBy("file_name")
    .orderBy(
        F.col(
            "bls_modified_at"
        ).desc_nulls_last(),

        F.col(
            "downloaded_at_utc"
        ).desc()
    )
)


latest_metadata = (
    metadata

    .filter(
        F.col("status") == "SUCCESS"
    )

    .withColumn(
        "rn",
        F.row_number().over(
            window_spec
        )
    )

    .filter(
        F.col("rn") == 1
    )

    .select(
        "file_name",
        "bls_modified_at"
    )
)


# ============================================================
# 9. CREATE PYTHON LOOKUP DICTIONARY
# ============================================================

previous_versions = {
    row["file_name"]:
        row["bls_modified_at"]

    for row in latest_metadata.collect()
}


print(
    f"Previously tracked successful files: "
    f"{len(previous_versions)}"
)


# ============================================================
# 10. READ BLS DIRECTORY
# ============================================================

print("\n" + "=" * 80)

print(
    "READING BLS DIRECTORY"
)

print("=" * 80)


directory_response = requests.get(
    BASE_URL,
    headers=HEADERS,
    timeout=60
)


directory_response.raise_for_status()


print(
    f"BLS directory HTTP status: "
    f"{directory_response.status_code}"
)


# ============================================================
# 11. PARSE BLS DIRECTORY
# ============================================================
#
# BLS uses an IIS-style directory listing rather than a
# standard HTML table.
#
# We extract:
#   - date
#   - time
#   - href
#   - filename
# ============================================================

pattern = re.compile(
    r"""
    (?P<date>
        \d{1,2}/\d{1,2}/\d{4}
    )
    \s+
    (?P<time>
        \d{1,2}:\d{2}\s+(?:AM|PM)
    )
    \s+
    \d+
    \s+
    <a\s+href=
    ["']
    (?P<href>[^"']+)
    ["']
    >
    (?P<filename>[^<]+)
    </a>
    """,
    re.IGNORECASE | re.VERBOSE
)


matches = pattern.finditer(
    directory_response.text
)


# ============================================================
# 12. BUILD SOURCE FILE LIST
# ============================================================

source_files = []


for match in matches:

    filename = match.group(
        "filename"
    ).strip()

    href = match.group(
        "href"
    ).strip()

    date_text = match.group(
        "date"
    )

    time_text = match.group(
        "time"
    )


    # --------------------------------------------------------
    # Only BLS Productivity files
    # --------------------------------------------------------

    if not filename.startswith("pr."):
        continue


    # --------------------------------------------------------
    # Parse BLS timestamp
    # --------------------------------------------------------

    timestamp_text = (
        f"{date_text} {time_text}"
    )


    try:

        bls_modified_at = datetime.strptime(
            timestamp_text,
            "%m/%d/%Y %I:%M %p"
        )

    except ValueError:

        bls_modified_at = None


    # --------------------------------------------------------
    # Build source record
    # --------------------------------------------------------

    source_files.append({

        "file_name":
            filename,

        "source_url":
            BASE_URL + filename,

        "bls_modified_at":
            bls_modified_at
    })


# ============================================================
# 13. REMOVE DUPLICATE FILENAMES
# ============================================================

unique_files = {}


for file_info in source_files:

    unique_files[
        file_info["file_name"]
    ] = file_info


source_files = sorted(
    unique_files.values(),
    key=lambda x: x["file_name"]
)


# ============================================================
# 14. DISPLAY DISCOVERED FILES
# ============================================================

print("\n" + "=" * 80)

print(
    "BLS FILES DISCOVERED"
)

print("=" * 80)


for file_info in source_files:

    print(
        f"{file_info['file_name']:30}"
        f" | "
        f"{file_info['bls_modified_at']}"
    )


print("=" * 80)


print(
    f"Total files discovered: "
    f"{len(source_files)}"
)


# ============================================================
# 15. SAFETY CHECK
# ============================================================
#
# Never proceed if the BLS directory unexpectedly returns
# zero matching files.
# ============================================================

if len(source_files) == 0:

    raise RuntimeError(
        "No BLS Productivity files were discovered. "
        "Ingestion stopped. "
        "Inspect directory_response.text to diagnose "
        "the BLS response."
    )


# ============================================================
# 16. DETERMINE FILE CHANGE STATUS
# ============================================================

files_to_download = []

files_to_skip = []


for file_info in source_files:

    filename = file_info[
        "file_name"
    ]

    current_modified = file_info[
        "bls_modified_at"
    ]

    previous_modified = previous_versions.get(
        filename
    )


    # --------------------------------------------------------
    # NEW FILE
    # --------------------------------------------------------

    if previous_modified is None:

        file_info[
            "change_type"
        ] = "NEW"

        files_to_download.append(
            file_info
        )


    # --------------------------------------------------------
    # UNKNOWN BLS TIMESTAMP
    # --------------------------------------------------------

    elif current_modified is None:

        file_info[
            "change_type"
        ] = "UNKNOWN"

        files_to_download.append(
            file_info
        )


    # --------------------------------------------------------
    # UPDATED FILE
    # --------------------------------------------------------

    elif current_modified > previous_modified:

        file_info[
            "change_type"
        ] = "UPDATED"

        files_to_download.append(
            file_info
        )


    # --------------------------------------------------------
    # UNCHANGED FILE
    # --------------------------------------------------------

    else:

        file_info[
            "change_type"
        ] = "UNCHANGED"

        files_to_skip.append(
            file_info
        )


# ============================================================
# 17. DISPLAY DOWNLOAD DECISION
# ============================================================

print("\n" + "=" * 80)

print(
    "DOWNLOAD DECISION"
)

print("=" * 80)


print(
    f"Files discovered : "
    f"{len(source_files)}"
)

print(
    f"Files to download: "
    f"{len(files_to_download)}"
)

print(
    f"Files to skip    : "
    f"{len(files_to_skip)}"
)

print("=" * 80)


# ------------------------------------------------------------
# Files to download
# ------------------------------------------------------------

for file_info in files_to_download:

    print(
        f"DOWNLOAD | "
        f"{file_info['change_type']:8} | "
        f"{file_info['file_name']:30} | "
        f"{file_info['bls_modified_at']}"
    )


# ------------------------------------------------------------
# Files to skip
# ------------------------------------------------------------

for file_info in files_to_skip:

    print(
        f"SKIP     | "
        f"{file_info['file_name']:30} | "
        f"{file_info['bls_modified_at']}"
    )


# ============================================================
# 18. INITIALIZE METADATA COLLECTION
# ============================================================

ingestion_metadata = []


# ============================================================
# 19. DOWNLOAD NEW / UPDATED FILES
# ============================================================

for file_info in files_to_download:

    filename = file_info["file_name"]

    source_url = file_info["source_url"]

    bls_modified_at = file_info["bls_modified_at"]

    change_type = file_info["change_type"]

    destination_path = (
        RAW_BLS_PATH + filename
    )


    print("\n" + "-" * 80)

    print(
        f"Downloading: {filename}"
    )

    print(
        f"Source URL : {source_url}"
    )

    print(
        f"Change type: {change_type}"
    )


    try:

        # ====================================================
        # 19.1 DOWNLOAD FROM BLS
        # ====================================================

        file_response = requests.get(
            source_url,
            headers=HEADERS,
            timeout=120
        )


        # ====================================================
        # 19.2 VALIDATE HTTP RESPONSE
        # ====================================================

        file_response.raise_for_status()


        # ====================================================
        # 19.3 HANDLE EXISTING UPDATED FILE
        # ====================================================
        #
        # For an UPDATED file, remove the previous version
        # before writing the new version.
        #
        # We do NOT use:
        #
        #   dbutils.fs.cp(..., overwrite=True)
        #
        # because that is not supported in this environment.
        # ====================================================

        if change_type == "UPDATED":

            try:

                dbutils.fs.rm(
                    destination_path
                )

                print(
                    f"Removed previous version: "
                    f"{filename}"
                )

            except Exception as remove_error:

                print(
                    f"Could not remove existing "
                    f"file: {remove_error}"
                )

                raise


        # ====================================================
        # 19.4 WRITE DIRECTLY TO UNITY CATALOG VOLUME
        # ====================================================
        #
        # IMPORTANT:
        # Do NOT use /tmp.
        #
        # Python writes directly to the Volume path.
        # ====================================================

        with open(
            destination_path,
            "wb"
        ) as f:

            f.write(
                file_response.content
            )


        # ====================================================
        # 19.5 VERIFY FILE EXISTS
        # ====================================================

        file_check = dbutils.fs.ls(
            destination_path
        )


        if not file_check:

            raise RuntimeError(
                f"File verification failed: "
                f"{destination_path}"
            )


        # ====================================================
        # 19.6 RECORD SUCCESS
        # ====================================================

        ingestion_metadata.append({

            "file_name":
                filename,

            "source_url":
                source_url,

            "bls_modified_at":
                bls_modified_at,

            "downloaded_at_utc":
                datetime.now(
                    timezone.utc
                ),

            "http_status":
                file_response.status_code,

            "status":
                "SUCCESS",

            "change_type":
                change_type,

            "error_message":
                None,

            "ingestion_run_id":
                ingestion_run_id
        })


        print(
            f"SUCCESS | "
            f"{change_type} | "
            f"{filename}"
        )


    except Exception as e:

        # ====================================================
        # RECORD FAILURE
        # ====================================================

        ingestion_metadata.append({

            "file_name":
                filename,

            "source_url":
                source_url,

            "bls_modified_at":
                bls_modified_at,

            "downloaded_at_utc":
                datetime.now(
                    timezone.utc
                ),

            "http_status":
                (
                    file_response.status_code
                    if "file_response" in locals()
                    else None
                ),

            "status":
                "FAILED",

            "change_type":
                change_type,

            "error_message":
                str(e),

            "ingestion_run_id":
                ingestion_run_id
        })


        print(
            f"FAILED | "
            f"{filename}"
        )

        print(
            f"Error: {str(e)}"
        )
        
# ============================================================
# 20. EXPLICIT METADATA SCHEMA
# ============================================================
#
# This prevents Spark from failing when error_message is
# NULL for every successful record.
# ============================================================

metadata_schema = StructType([

    StructField(
        "file_name",
        StringType(),
        True
    ),

    StructField(
        "source_url",
        StringType(),
        True
    ),

    StructField(
        "bls_modified_at",
        TimestampType(),
        True
    ),

    StructField(
        "downloaded_at_utc",
        TimestampType(),
        True
    ),

    StructField(
        "http_status",
        IntegerType(),
        True
    ),

    StructField(
        "status",
        StringType(),
        True
    ),

    StructField(
        "change_type",
        StringType(),
        True
    ),

    StructField(
        "error_message",
        StringType(),
        True
    ),

    StructField(
        "ingestion_run_id",
        StringType(),
        True
    )
])


# ============================================================
# 21. WRITE INGESTION METADATA
# ============================================================

if ingestion_metadata:

    new_metadata_df = spark.createDataFrame(
        ingestion_metadata,
        schema=metadata_schema
    )


    # --------------------------------------------------------
    # Explicit column order
    # --------------------------------------------------------

    new_metadata_df = (
        new_metadata_df
        .select(
            "file_name",
            "source_url",
            "bls_modified_at",
            "downloaded_at_utc",
            "http_status",
            "status",
            "change_type",
            "error_message",
            "ingestion_run_id"
        )
    )


    # --------------------------------------------------------
    # Append to Delta table
    # --------------------------------------------------------

    (
        new_metadata_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(METADATA_TABLE)
    )


    print(
        "\nMetadata records written: "
        f"{len(ingestion_metadata)}"
    )


else:

    print(
        "\nNo files downloaded."
    )

    print(
        "No new metadata records added."
    )


# ============================================================
# 22. CALCULATE INGESTION SUMMARY
# ============================================================

successful = sum(
    1
    for x in ingestion_metadata
    if x["status"] == "SUCCESS"
)


failed = sum(
    1
    for x in ingestion_metadata
    if x["status"] == "FAILED"
)


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)

print(
    "BLS INGESTION COMPLETE"
)

print("=" * 80)

print(
    f"Ingestion run ID : "
    f"{ingestion_run_id}"
)

print(
    f"Files discovered : "
    f"{len(source_files)}"
)

print(
    f"Files downloaded : "
    f"{successful}"
)

print(
    f"Files skipped    : "
    f"{len(files_to_skip)}"
)

print(
    f"Files failed     : "
    f"{failed}"
)

print("=" * 80)


# ============================================================
# 24. DISPLAY LATEST INGESTION METADATA
# ============================================================

print(
    "\nLatest ingestion metadata:"
)


display(
    spark.sql(f"""
        SELECT
            file_name,
            bls_modified_at,
            downloaded_at_utc,
            status,
            change_type,
            http_status,
            error_message,
            ingestion_run_id
        FROM {METADATA_TABLE}
        ORDER BY
            downloaded_at_utc DESC,
            file_name
    """)
)

Column already exists: change_type
Metadata table ready: bls_dataquest.bronze.ingestion_metadata
Ingestion run ID: 20260912041412854349
Previously tracked successful files: 12

READING BLS DIRECTORY
BLS directory HTTP status: 200

BLS FILES DISCOVERED
pr.class                       | 2026-09-03 08:30:00
pr.contacts                    | 2022-09-13 16:52:00
pr.data.0.Current              | 2026-09-03 08:30:00
pr.data.1.AllData              | 2026-09-03 08:30:00
pr.duration                    | 2026-09-03 08:30:00
pr.footnote                    | 2026-09-03 08:30:00
pr.measure                     | 2026-09-03 08:30:00
pr.period                      | 1994-01-07 15:53:00
pr.seasonal                    | 2011-11-18 16:05:00
pr.sector                      | 2026-09-03 08:30:00
pr.series                      | 2026-09-03 08:30:00
pr.txt                         | 2011-11-17 17:11:00
Total files discovered: 12

DOWNLOAD DECISION
Files discovered : 12
Files to download: 0
Files to skip    : 12
S

file_name,bls_modified_at,downloaded_at_utc,status,change_type,http_status,error_message,ingestion_run_id
pr.txt,2011-11-17T17:11:00.000Z,2026-09-12T04:13:34.222Z,SUCCESS,NEW,200,null,20260912041324573649
pr.series,2026-09-03T08:30:00.000Z,2026-09-12T04:13:33.542Z,SUCCESS,NEW,200,null,20260912041324573649
pr.sector,2026-09-03T08:30:00.000Z,2026-09-12T04:13:32.898Z,SUCCESS,NEW,200,null,20260912041324573649
pr.seasonal,2011-11-18T16:05:00.000Z,2026-09-12T04:13:32.323Z,SUCCESS,NEW,200,null,20260912041324573649
pr.period,1994-01-07T15:53:00.000Z,2026-09-12T04:13:31.700Z,SUCCESS,NEW,200,null,20260912041324573649
pr.measure,2026-09-03T08:30:00.000Z,2026-09-12T04:13:31.132Z,SUCCESS,NEW,200,null,20260912041324573649
pr.footnote,2026-09-03T08:30:00.000Z,2026-09-12T04:13:30.504Z,SUCCESS,NEW,200,null,20260912041324573649
pr.duration,2026-09-03T08:30:00.000Z,2026-09-12T04:13:29.904Z,SUCCESS,NEW,200,null,20260912041324573649
pr.data.1.AllData,2026-09-03T08:30:00.000Z,2026-09-12T04:13:29.269Z,SUCCESS,NEW,200,null,20260912041324573649
pr.data.0.Current,2026-09-03T08:30:00.000Z,2026-09-12T04:13:28.538Z,SUCCESS,NEW,200,null,20260912041324573649
